In [22]:
%load_ext autoreload
%autoreload 2

import sklearn
import scipy 
import numpy as np
import pandas as pd
import os
import sys

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
sys.path.append(os.path.abspath("src"))
import jdcoot
import math
from scipy.io import loadmat

from jdcoot.models.discrete_unsupervised_jdcoot import discrete_unsupervised_jdcoot
from jdcoot.models.discrete_semisupervised_jdcoot import discrete_semisupervised_jdcoot
from jdcoot.models.discrete_partial_jdcoot import discrete_partial_jdcoot
from jdcoot.models.discrete_unsupervised_coot import discrete_unsupervised_coot
from jdcoot.models.discrete_semisupervised_coot import discrete_semisupervised_coot
from jdcoot.models.discrete_partial_coot import discrete_partial_coot
from jdcoot.models.discrete_semisupervised_reference import discrete_semisupervised_reference
from jdcoot.models.discrete_partial_reference import discrete_partial_reference


S_data = pd.read_csv("data_omique_S.csv")
T_data = pd.read_csv("data_omique_T.csv")
X_colsS = ['X'+str(i) for i in range(S_data.shape[1]-1)]
X_colsT = ['X'+str(i) for i in range(T_data.shape[1]-1)]
S_data.columns = X_colsS + ['Z']
T_data.columns = X_colsT + ['Z']

source = S_data
target = T_data

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
source.shape, target.shape

((79, 60661), (79, 370366))

In [2]:
target

,X0,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X370356,X370357,X370358,X370359,X370360,X370361,X370362,X370363,X370364,Z
0,0.331337,0.433665,0.907527,0.929226,0.803080,0.913378,0.915925,0.975695,0.943965,0.562206,...,0.014883,0.051429,0.030359,0.090640,0.081120,0.102101,0.066396,0.455526,0.949053,2.0
1,0.380037,0.194028,0.644863,0.483128,0.415208,0.671466,0.497068,0.964929,0.877985,0.621223,...,0.398857,0.258503,0.278820,0.208904,0.288537,0.533426,0.172734,0.254140,0.936177,0.0
2,0.857103,0.819285,0.924852,0.903546,0.822678,0.950027,0.940995,0.971426,0.946664,0.910891,...,0.157011,0.154011,0.175260,0.227087,0.111708,0.171197,0.073877,0.180601,0.960520,1.0
3,0.797363,0.643394,0.932961,0.954545,0.938442,0.949231,0.953950,0.975893,0.953283,0.945784,...,0.012174,0.060258,0.027889,0.090082,0.052156,0.068259,0.036900,0.283432,0.960715,1.0
4,0.848468,0.229679,0.499823,0.784611,0.761054,0.883729,0.923103,0.868607,0.880567,0.926299,...,0.482362,0.191811,0.422333,0.467804,0.149098,0.712462,0.503727,0.416620,0.931660,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,0.417972,0.733041,0.830049,0.795599,0.364169,0.856912,0.847503,0.969288,0.878032,0.896617,...,0.013409,0.072668,0.026789,0.090064,0.055967,0.086504,0.034187,0.081877,0.959424,0.0
75,0.892024,0.775338,0.936082,0.917142,0.889666,0.930431,0.782313,0.937133,0.954438,0.952979,...,0.014994,0.073096,0.031776,0.101467,0.059425,0.091796,0.031106,0.197543,0.971323,1.0
76,0.686657,0.176965,0.858470,0.562835,0.244272,0.907377,0.912395,0.969669,0.949788,0.927164,...,0.462350,0.483187,0.494165,0.252890,0.602378,0.702331,0.478408,0.349309,0.877274,0.0
77,0.934454,0.727729,0.879593,0.865560,0.724130,0.902862,0.934276,0.970666,0.965207,0.913717,...,0.013428,0.070319,0.026220,0.092631,0.063690,0.071169,0.039468,0.165058,0.959352,2.0


In [24]:
source = S_data
target = T_data
source = source.dropna(subset=['Z'])

target = source.dropna(subset=['Z'])

np.unique(source.Z)

source.shape, target.shape

((78, 60661), (78, 60661))

In [73]:
from jdcoot.utils import xcolumns
from sklearn.feature_selection import VarianceThreshold
source = S_data
target = T_data
source = source.dropna(subset=['Z'])
target = target.dropna(subset=['Z'])
x_source = source.loc[:, xcolumns(source)].values
x_target = target.loc[:, xcolumns(target)].values
#selector = VarianceThreshold(threshold=0.5)
selector = VarianceThreshold(threshold=0.3)
x_source_reduced = selector.fit_transform(x_source)
selector = VarianceThreshold(threshold=0.12)
#selector = VarianceThreshold(threshold=0.1)
x_target_reduced = selector.fit_transform(x_target)
x_target_reduced.shape
X_df = pd.DataFrame(x_source_reduced, columns=['X'+str(i) for i in range(x_source_reduced.shape[1])])
X_df['Z'] = source['Z'].values
source = X_df
X_df = pd.DataFrame(x_target_reduced, columns=['X'+str(i) for i in range(x_target_reduced.shape[1])])
X_df['Z'] = target['Z'].values
target = X_df
source.shape, target.shape

((78, 1788), (78, 1078))

In [51]:
target

,X0,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X60512,X60513,X60514,X60515,X60516,X60517,X60518,X60519,X60520,Z
0,0.433665,0.803080,0.646837,0.843355,0.877903,0.798815,0.982415,0.434924,0.018803,0.032013,...,0.863415,0.730676,0.016662,0.019008,0.016776,0.047850,0.025646,0.049827,0.204032,2.0
1,0.194028,0.415208,0.116847,0.249093,0.366477,0.096748,0.329840,0.176833,0.013811,0.023012,...,0.562009,0.575389,0.588001,0.532977,0.641690,0.539211,0.174914,0.758622,0.519730,0.0
2,0.819285,0.822678,0.117841,0.241890,0.283895,0.219936,0.971640,0.154427,0.015414,0.024485,...,0.928428,0.947320,0.132949,0.179450,0.168547,0.221971,0.076037,0.075427,0.882865,1.0
3,0.643394,0.938442,0.867871,0.858461,0.930585,0.857678,0.988243,0.576937,0.015632,0.031125,...,0.964975,0.230504,0.015594,0.013431,0.014899,0.062020,0.018839,0.030643,0.231643,1.0
4,0.229679,0.761054,0.207326,0.471554,0.538779,0.068366,0.649715,0.203437,0.015389,0.023411,...,0.310373,0.616332,0.652140,0.589901,0.570310,0.594290,0.578368,0.611783,0.616979,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,0.733041,0.364169,0.266076,0.422501,0.486601,0.081517,0.508009,0.306164,0.016815,0.028808,...,0.957583,0.492130,0.016697,0.017669,0.017898,0.056328,0.022025,0.036220,0.314378,0.0
74,0.775338,0.889666,0.875366,0.849793,0.582354,0.870951,0.977635,0.848177,0.015396,0.028348,...,0.165217,0.941340,0.014586,0.016810,0.017425,0.063671,0.017832,0.028403,0.138123,1.0
75,0.176965,0.244272,0.146807,0.306021,0.367064,0.058455,0.974611,0.146063,0.014086,0.025156,...,0.376853,0.307817,0.539658,0.536321,0.379427,0.573468,0.449295,0.672339,0.512420,0.0
76,0.727729,0.724130,0.705737,0.634697,0.613247,0.722534,0.979706,0.681679,0.703638,0.726884,...,0.955416,0.957125,0.014038,0.015399,0.015365,0.056222,0.021011,0.033257,0.129832,2.0


In [74]:
a = np.random.choice(np.arange(len(source)), math.ceil(0.9 * len(source)), replace=False)
b = np.random.choice(np.arange(len(target)), math.ceil(0.9 * len(target)), replace=False)

S_test = source.iloc[np.setdiff1d(np.arange(len(source)), a), :].reset_index(drop=True)
T_test = target.iloc[np.setdiff1d(np.arange(len(target)), b), :].reset_index(drop=True)
S = source.iloc[a, :].reset_index(drop=True)
T = target.iloc[b, :].reset_index(drop=True)
import os
os.environ["PYTHONHASHSEED"] = "42"

import numpy as np
import random
import tensorflow as tf



In [ ]:
#Test predicteur

import numpy as np
from sklearn.preprocessing import OneHotEncoder as onehot
from sklearn.model_selection import train_test_split
from jdcoot.utils import xcolumns, discrete_classifiers, discrete_accuracy


source_levels = np.sort(np.unique(S.Z))
target_levels = np.sort(np.unique(T.Z))

nClass = len(np.union1d(source_levels, target_levels))
categories = [np.arange(nClass)]

enc = onehot(handle_unknown="ignore", sparse_output=False, categories=categories)

x_source_train = S.loc[:, xcolumns(S)].values
z_source_train = enc.fit_transform(S.Z.values[:, np.newaxis])

x_target_train = T.loc[:, xcolumns(T)].values
z_target_train = enc.fit_transform(T.Z.values[:, np.newaxis])

x_source_test = S_test.loc[:, xcolumns(S_test)].values
z_source_test = S_test.Z.values

x_target_test = T_test.loc[:, xcolumns(T_test)].values
z_target_test = T_test.Z.values

clf_source, clf_target = discrete_classifiers(S, T, "relu", "softmax")


clf_target.fit(x_target_train, z_target_train, batch_size=len(x_target_train), epochs=10, verbose=0, shuffle=False)

clf_source.fit(x_source_train, z_source_train, batch_size=len(x_source_train), epochs=10, verbose=0, shuffle=False)

z_target_pred = enc.inverse_transform(
        clf_target.predict(x_target_test, verbose=0)
    ).ravel()
z_source_pred = enc.inverse_transform(
        clf_source.predict(x_source_test, verbose=0)
    ).ravel()

perf_test_source = discrete_accuracy(z_source_pred,S_test.Z)
perf_test_target = discrete_accuracy(z_target_pred, T_test.Z)


In [76]:
perf_test_source

np.float64(0.8571428571428571)

In [77]:
perf_test_target

np.float64(0.8571428571428571)

In [78]:
import numpy as np
import math
from sklearn.preprocessing import OneHotEncoder as onehot
from jdcoot.utils import xcolumns, discrete_classifiers, discrete_accuracy

n_runs = 10

perf_source_list = []
perf_target_list = []

for i in range(n_runs):

    a = np.random.choice(np.arange(len(source)), math.ceil(0.9 * len(source)), replace=False)
    b = np.random.choice(np.arange(len(target)), math.ceil(0.9 * len(target)), replace=False)

    S_test = source.iloc[np.setdiff1d(np.arange(len(source)), a), :].reset_index(drop=True)
    T_test = target.iloc[np.setdiff1d(np.arange(len(target)), b), :].reset_index(drop=True)

    S = source.iloc[a, :].reset_index(drop=True)
    T = target.iloc[b, :].reset_index(drop=True)

    source_levels = np.sort(np.unique(S.Z))
    target_levels = np.sort(np.unique(T.Z))

    nClass = len(np.union1d(source_levels, target_levels))
    categories = [np.arange(nClass)]

    enc = onehot(handle_unknown="ignore", sparse_output=False, categories=categories)

    x_source_train = S.loc[:, xcolumns(S)].values
    z_source_train = enc.fit_transform(S.Z.values[:, np.newaxis])

    x_target_train = T.loc[:, xcolumns(T)].values
    z_target_train = enc.fit_transform(T.Z.values[:, np.newaxis])

    x_source_test = S_test.loc[:, xcolumns(S_test)].values
    z_source_test = S_test.Z.values

    x_target_test = T_test.loc[:, xcolumns(T_test)].values
    z_target_test = T_test.Z.values

    clf_source, clf_target = discrete_classifiers(S, T, "relu", "softmax")

    clf_target.fit(x_target_train, z_target_train,
                   batch_size=len(x_target_train), epochs=10, verbose=0, shuffle=False)

    clf_source.fit(x_source_train, z_source_train,
                   batch_size=len(x_source_train), epochs=10, verbose=0, shuffle=False)

    z_target_pred = enc.inverse_transform(
        clf_target.predict(x_target_test, verbose=0)
    ).ravel()

    z_source_pred = enc.inverse_transform(
        clf_source.predict(x_source_test, verbose=0)
    ).ravel()

    perf_test_source = discrete_accuracy(z_source_pred, z_source_test)
    perf_test_target = discrete_accuracy(z_target_pred, z_target_test)

    perf_source_list.append(perf_test_source)
    perf_target_list.append(perf_test_target)


mean_source = np.mean(perf_source_list)
mean_target = np.mean(perf_target_list)

print("Mean source test accuracy:", mean_source)
print("Mean target test accuracy:", mean_target)

Mean source test accuracy: 0.7142857142857142
Mean target test accuracy: 0.8285714285714285


In [16]:
results = []
numRepetitions = 1
#algo = "sinkhorn"
algo = "emd"
reg = 0.1

In [20]:
#Test coot
seed = 42

np.random.seed(seed)
random.seed(seed)
tf.random.set_seed(seed)
algo = "sinkhorn"
reg = 0.1
pure_source, pure_target, test_source, test_target =  discrete_unsupervised_coot(S, T, S_test, T_test,algo=algo,reg=reg,batch_size=len(x_source_train))
test_target



Delta:       0.0355328 	 Loss:       1.0887392
Delta:       0.0369520 	 Loss:       1.0688088
Delta:       0.0298457 	 Loss:       1.0594352
Delta:       0.0224613 	 Loss:       1.0548969
Delta:       0.0179421 	 Loss:       1.0527141
Delta:       0.0134685 	 Loss:       1.0515302
Delta:       0.0117900 	 Loss:       1.0508245
Delta:       0.0120477 	 Loss:       1.0502625
Delta:       0.0129738 	 Loss:       1.0495233
Delta:       0.0136951 	 Loss:       1.0485801
Delta:       0.0134978 	 Loss:       1.0475085
Delta:       0.0105706 	 Loss:       1.0467099
Delta:       0.0074890 	 Loss:       1.0462968
Delta:       0.0057166 	 Loss:       1.0460630
Delta:       0.0045449 	 Loss:       1.0459442
Delta:       0.0033655 	 Loss:       1.0458764
Delta:       0.0034922 	 Loss:       1.0458472
Delta:       0.0025385 	 Loss:       1.0458154
Delta:       0.0016386 	 Loss:       1.0457861
Delta:       0.0021867 	 Loss:       1.0457812
Delta:       0.0023173 	 Loss:       1.0457793
Delta:       

np.float64(0.5714285714285714)

In [22]:
seed = 42

np.random.seed(seed)
random.seed(seed)
tf.random.set_seed(seed)
algo = "emd"
reg = 0.1
pure_source, pure_target, test_source, test_target =  discrete_unsupervised_coot(T, S, T_test, S_test,algo=algo,reg=reg,batch_size=len(x_source_train))
test_target


Delta:       0.1462019 	 Loss:       0.9408186
Delta:       0.0000000 	 Loss:       0.9408186
converged at iter  1


np.float64(0.5714285714285714)

In [ ]:
alpha= 0.5# hyperparamètre devant la loss a été optimé
algo = "emd"
reg = 0.1
pure_source, pure_target, test_source, test_target = discrete_unsupervised_jdcoot(S, T, S_test, T_test,algo=algo,reg=reg,batch_size=len(x_source_train), alpha=alpha)
test_target

Delta: 0.09227688251480859 	  Loss: 0.8915382451109897 	 Accuracy: 0.4444444444444444
Delta: 0.07923423515483474 	  Loss: 0.8791124123921206 	 Accuracy: 0.4444444444444444
Delta: 0.07170577576639313 	  Loss: 0.8651861422008841 	 Accuracy: 0.4444444444444444
Delta: 0.0 	  Loss: 0.8651861422008841 	 Accuracy: 0.4444444444444444
converged at iter  3


np.float64(0.7333333333333333)

In [46]:
alpha=1.5
algo = "emd"
reg = 0.1

pure_source, pure_target, test_source, test_target = discrete_unsupervised_jdcoot(T, S, T_test, S_test, algo=algo, reg=reg,batch_size=len(x_source_train), alpha=alpha)
test_target

Delta: 0.0 	  Loss: 0.9186105615795105 	 Accuracy: 0.4126984126984127
converged at iter  0


np.float64(0.4)

In [57]:
S = source
T = target


#Test predicteur

tf.config.experimental.enable_op_determinism()
source_levels = np.sort(np.unique(S.Z))
target_levels = np.sort(np.unique(T.Z))

nClass = len(np.union1d(source_levels, target_levels))
categories = [np.arange(nClass)]

enc = onehot(handle_unknown="ignore", sparse_output=False, categories=categories)

x_source_train = S.loc[:, xcolumns(S)].values
z_source_train = enc.fit_transform(S.Z.values[:, np.newaxis])

x_target_train = T.loc[:, xcolumns(T)].values
z_target_train = enc.fit_transform(T.Z.values[:, np.newaxis])

x_source_test = S.loc[:, xcolumns(S)].values
z_source_test = S.Z.values

x_target_test = T.loc[:, xcolumns(T)].values
z_target_test = T.Z.values

clf_source, clf_target = discrete_classifiers(S, T, "relu", "softmax")


clf_target.fit(x_target_train, z_target_train, batch_size=len(x_target_train), epochs=10, verbose=0, shuffle=False)

clf_source.fit(x_source_train, z_source_train, batch_size=len(x_source_train), epochs=10, verbose=0, shuffle=False)

z_target_pred = enc.inverse_transform(
        clf_target.predict(x_target_test, verbose=0)
    ).ravel()
z_source_pred = enc.inverse_transform(
        clf_source.predict(x_source_test, verbose=0)
    ).ravel()

perf_test_source = discrete_accuracy(z_source_pred,S.Z)
perf_test_target = discrete_accuracy(z_target_pred, T.Z)


In [59]:
perf_test_target

np.float64(0.9871794871794872)

In [68]:
algo = "emd"
reg = 0.1
pure_source, pure_target, test_source, test_target =  discrete_unsupervised_coot(S, T, S, T,algo=algo,reg=reg,batch_size=len(x_source_train))
test_target


Delta:       0.1570430 	 Loss:      -0.0000000
Delta:       0.0000000 	 Loss:      -0.0000000
converged at iter  1


np.float64(0.9487179487179487)

In [69]:
pure_source, pure_target, test_source, test_target =  discrete_unsupervised_coot(T, S, T, S,algo=algo,reg=reg,batch_size=len(x_source_train))
test_target

Delta:       0.1570430 	 Loss:      -0.0000000
Delta:       0.0000000 	 Loss:      -0.0000000
converged at iter  1


np.float64(0.9358974358974359)

In [66]:
alpha= 1.5
seed = 42

np.random.seed(seed)
random.seed(seed)
tf.random.set_seed(seed)
pure_source, pure_target, test_source, test_target = discrete_unsupervised_jdcoot(S, T, S, T, algo=algo, reg=reg,batch_size=len(x_source_train), alpha=alpha)
test_target

Delta: 0.0 	  Loss: -1.1477454727735212e-17 	 Accuracy: 1.0
converged at iter  0


np.float64(1.0)

In [67]:
alpha= 1.5
seed = 42

np.random.seed(seed)
random.seed(seed)
tf.random.set_seed(seed)
pure_source, pure_target, test_source, test_target = discrete_unsupervised_jdcoot(T, S, T, S, algo=algo, reg=reg,batch_size=len(x_source_train), alpha=alpha)
test_target

Delta: 0.0 	  Loss: -1.1477454727735212e-17 	 Accuracy: 1.0
converged at iter  0


np.float64(1.0)

In [70]:
np.linspace(0, 2, 21)

array([0. , 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1. , 1.1, 1.2,
       1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2. ])

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
import math

alpha_values = np.linspace(0, 2, 21)
num_folds = 5

kf_source = KFold(n_splits=num_folds, shuffle=True, random_state=42)
kf_target = KFold(n_splits=num_folds, shuffle=True, random_state=42)

results = []

algo = "sinkhorn"
reg = 0.1

best_alpha_global = None
best_score_global = -np.inf

for alpha in alpha_values:
    
    fold_scores = []
    
    for (train_idx_s, test_idx_s), (train_idx_t, test_idx_t) in zip(
        kf_source.split(source), 
        kf_target.split(target)
    ):
        
        S = source.iloc[train_idx_s].reset_index(drop=True)
        S_test = source.iloc[test_idx_s].reset_index(drop=True)

        T = target.iloc[train_idx_t].reset_index(drop=True)
        T_test = target.iloc[test_idx_t].reset_index(drop=True)

        pure_source, pure_target, test_source, test_target = \
            discrete_unsupervised_jdcoot(
                S, T,
                S_test, T_test,
                algo=algo,
                reg=reg,
                batch_size=len(S),
                alpha=alpha
            )

        score = test_target   

        fold_scores.append(score)

    mean_score = np.mean(fold_scores)

    if mean_score > best_score_global:
        best_score_global = mean_score
        best_alpha_global = alpha

    results.append({
        "alpha": alpha,
        "mean_score": mean_score
    })

df_results = pd.DataFrame(results)

print("Best alpha:", best_alpha_global)
print("Best score:", best_score_global)

df_results.to_excel("results_cv.xlsx", index=False)

Delta: 0.03859112163244333 	  Loss: 1.0757085724823683 	 Accuracy: 0.6129032258064516
Delta: 0.030214226262528694 	  Loss: 1.0650784225153846 	 Accuracy: 0.5
Delta: 0.022991820540486235 	  Loss: 1.0606929853856388 	 Accuracy: 0.5483870967741935
Delta: 0.020983479227586723 	  Loss: 1.0581411542976906 	 Accuracy: 0.5645161290322581
Delta: 0.019418545181207312 	  Loss: 1.0556429188762637 	 Accuracy: 0.5806451612903226
Delta: 0.01695762847509114 	  Loss: 1.0536094453216345 	 Accuracy: 0.5483870967741935
Delta: 0.011256960611149863 	  Loss: 1.05261203162696 	 Accuracy: 0.5967741935483871
Delta: 0.0071649772076922795 	  Loss: 1.0522742876953193 	 Accuracy: 0.5483870967741935
Delta: 0.005925570069823322 	  Loss: 1.0521076163809995 	 Accuracy: 0.5483870967741935
Delta: 0.005867788978077584 	  Loss: 1.051991302606024 	 Accuracy: 0.5483870967741935
Delta: 0.0049383469305281785 	  Loss: 1.0518938571144967 	 Accuracy: 0.5483870967741935
Delta: 0.003986628174880538 	  Loss: 1.0518017552229963 	 Acc

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.003856427095219662 	  Loss: 1.0255004149162632 	 Accuracy: 0.4032258064516129
Delta: 0.003191977712695043 	  Loss: 1.02544191966816 	 Accuracy: 0.4032258064516129
Delta: 0.0027925983129557437 	  Loss: 1.0254247182382745 	 Accuracy: 0.4032258064516129
Delta: 0.002478626994706039 	  Loss: 1.0254045005318275 	 Accuracy: 0.4032258064516129
Delta: 0.0018390018905339845 	  Loss: 1.0253673802518857 	 Accuracy: 0.4032258064516129
Delta: 0.0021851059057896405 	  Loss: 1.0253420130654989 	 Accuracy: 0.4032258064516129
Delta: 0.001593525548429793 	  Loss: 1.0253177371650182 	 Accuracy: 0.4032258064516129
Delta: 0.0021047847208479206 	  Loss: 1.0253087279184403 	 Accuracy: 0.4032258064516129
Delta: 0.0014211017568431588 	  Loss: 1.025277792577484 	 Accuracy: 0.4032258064516129
Delta: 0.0015651804954751286 	  Loss: 1.025240460530199 	 Accuracy: 0.4032258064516129
Delta: 0.0018042620079034277 	  Loss: 1.0252184341573476 	 Accuracy: 0.4032258064516129
Delta: 0.0014317042240135263 	  Loss: 1.

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.0015087981733247881 	  Loss: 1.0198074246722078 	 Accuracy: 0.5806451612903226
Delta: 0.001051631663777336 	  Loss: 1.0197827144039855 	 Accuracy: 0.5806451612903226
Delta: 0.0008549233313116429 	  Loss: 1.0197640142658957 	 Accuracy: 0.5806451612903226
Delta: 0.0009677726313131661 	  Loss: 1.0197485359595524 	 Accuracy: 0.5806451612903226
Delta: 0.00106421944152789 	  Loss: 1.0197366188242818 	 Accuracy: 0.5806451612903226
Delta: 0.0010825021175918362 	  Loss: 1.0197414264616629 	 Accuracy: 0.5806451612903226
Delta: 0.001091871738340018 	  Loss: 1.0197396894053465 	 Accuracy: 0.5806451612903226
Delta: 0.001016422869391547 	  Loss: 1.0197181709441443 	 Accuracy: 0.5806451612903226
Delta: 0.0009599827107271506 	  Loss: 1.0197026167001328 	 Accuracy: 0.5806451612903226
Delta: 0.0007950640925560006 	  Loss: 1.0196933139095683 	 Accuracy: 0.5806451612903226
Delta: 0.0002863731975095828 	  Loss: 1.0196798998236716 	 Accuracy: 0.5806451612903226
Delta: 0.0002594045783474038 	  Loss:

...

Delta: 0.021199872426309736 	  Loss: 1.0466401829954324 	 Accuracy: 0.2857142857142857
Delta: 0.014725952318069934 	  Loss: 1.0447429745033738 	 Accuracy: 0.2857142857142857
Delta: 0.010833137602887177 	  Loss: 1.043912127676025 	 Accuracy: 0.2857142857142857
Delta: 0.010509294840460643 	  Loss: 1.043280811862307 	 Accuracy: 0.2857142857142857
Delta: 0.009771363205965704 	  Loss: 1.0426547868748064 	 Accuracy: 0.2857142857142857
Delta: 0.009162986915619495 	  Loss: 1.0420583129172618 	 Accuracy: 0.2857142857142857
Delta: 0.008519067615004315 	  Loss: 1.041590907136221 	 Accuracy: 0.2857142857142857
Delta: 0.007380583269763616 	  Loss: 1.0412101519335766 	 Accuracy: 0.2857142857142857
Delta: 0.006940937269168876 	  Loss: 1.0409411940753786 	 Accuracy: 0.2857142857142857
Delta: 0.006606061516962339 	  Loss: 1.0406845551946067 	 Accuracy: 0.2857142857142857
Delta: 0.00585154100816168 	  Loss: 1.0404367374086545 	 Accuracy: 0.2857142857142857
Delta: 0.005626711907607029 	  Loss: 1.04031386

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.032291209567883686 	  Loss: 1.068315428182944 	 Accuracy: 0.3709677419354839
Delta: 0.02441466095665889 	  Loss: 1.0641073093420177 	 Accuracy: 0.3709677419354839
Delta: 0.020702142359131186 	  Loss: 1.0610776740779964 	 Accuracy: 0.3709677419354839
Delta: 0.016888047146464344 	  Loss: 1.0589469398185165 	 Accuracy: 0.3709677419354839
Delta: 0.013385577961643145 	  Loss: 1.05761659433694 	 Accuracy: 0.3709677419354839
Delta: 0.010519744992050998 	  Loss: 1.056845591729574 	 Accuracy: 0.3709677419354839
Delta: 0.008701271189771462 	  Loss: 1.0564215132416477 	 Accuracy: 0.3709677419354839
Delta: 0.008414244070216568 	  Loss: 1.05606895179137 	 Accuracy: 0.3709677419354839
Delta: 0.007750528101181223 	  Loss: 1.0557484203494258 	 Accuracy: 0.3709677419354839
Delta: 0.007568504598536353 	  Loss: 1.0554798402591412 	 Accuracy: 0.3709677419354839
Delta: 0.006696640355341052 	  Loss: 1.0552720538949898 	 Accuracy: 0.3709677419354839
Delta: 0.006999877074711235 	  Loss: 1.05509501302

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.028266341127272446 	  Loss: 1.0646005634545481 	 Accuracy: 0.06451612903225806
Delta: 0.021691111507961935 	  Loss: 1.0615607360240829 	 Accuracy: 0.06451612903225806
Delta: 0.01811490989844219 	  Loss: 1.0596557252068066 	 Accuracy: 0.06451612903225806
Delta: 0.01551847766147842 	  Loss: 1.0584422496900237 	 Accuracy: 0.06451612903225806
Delta: 0.015150033105038298 	  Loss: 1.0575215032227867 	 Accuracy: 0.06451612903225806
Delta: 0.01406432182987655 	  Loss: 1.056603766779959 	 Accuracy: 0.06451612903225806
Delta: 0.012494642274081846 	  Loss: 1.0558633907723776 	 Accuracy: 0.06451612903225806
Delta: 0.01137818859663781 	  Loss: 1.055364018278855 	 Accuracy: 0.06451612903225806
Delta: 0.010054555105884624 	  Loss: 1.0549354411711422 	 Accuracy: 0.06451612903225806
Delta: 0.0094043752931877 	  Loss: 1.054616322590187 	 Accuracy: 0.06451612903225806
Delta: 0.008600649140687968 	  Loss: 1.0544342674831966 	 Accuracy: 0.06451612903225806
Delta: 0.0073385188653712135 	  Loss: 1.0

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

numRepetitions = 10
num_folds = 5

algo = "emd"
reg = 0.1
alpha = 0.6   # fixe ici si tu ne fais pas tuning

results = []

for repe in range(numRepetitions):
    print("num repe :", repe + 1)

    kf_source = KFold(n_splits=num_folds, shuffle=True, random_state=repe)
    kf_target = KFold(n_splits=num_folds, shuffle=True, random_state=repe)

    for fold, ((train_s, test_s), (train_t, test_t)) in enumerate(
        zip(kf_source.split(source), kf_target.split(target))
    ):

        S = source.iloc[train_s].reset_index(drop=True)
        S_test = source.iloc[test_s].reset_index(drop=True)

        T = target.iloc[train_t].reset_index(drop=True)
        T_test = target.iloc[test_t].reset_index(drop=True)

        # ==========================
        # COOT
        # ==========================
        pure_source, pure_target, test_source, test_target = \
            discrete_unsupervised_coot(
                S, T, S_test, T_test,
                algo=algo,
                reg=reg,
                batch_size=len(S)
            )

        score_coot = test_target  # ⚠️ mets ta vraie métrique

        results.append({
            "repetition": repe,
            "fold": fold,
            "recoding": "coot",
            "learning": "unsupervised",
            "score": score_coot,
        })

        # ==========================
        # JDCOOT
        # ==========================
        pure_source, pure_target, test_source, test_target = \
            discrete_unsupervised_jdcoot(
                S, T, S_test, T_test,
                algo=algo,
                reg=reg,
                batch_size=len(S),
                alpha=alpha
            )

        score_jdcoot = test_target  # ⚠️ mets ta vraie métrique

        results.append({
            "repetition": repe,
            "fold": fold,
            "recoding": "jdcoot",
            "learning": "unsupervised",
            "score": score_jdcoot,
        })

# ==============================
# Résumé
# ==============================

df_results = pd.DataFrame(results)

df_summary = (
    df_results
    .groupby(["recoding"])
    .agg(mean_score=("score", "mean"),
         std_score=("score", "std"))
    .reset_index()
)

print(df_summary)

df_summary.to_excel("results_cv_mean.xlsx", index=False)

num repe : 1
